# Approach: Model 6 - Complete collaboration with addition of:
- HT-MIA
- Min-K%++
- Extended K values
- Token-level LLR
- Soft Voting Ensemble

In [ ]:
import sys
# Path to the model6 directory after uploading to Colab.
# If you uploaded to /content/, this is correct.
sys.path.insert(0, '/content/model6')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/model6/"

In [ ]:
# imports

import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch
from models import load_data, load_models_and_tokenizer
from batches import process_dataset_in_batches, save_features, load_features
from build_membership_classifier import build_membership_classifier, compute_zlib_baseline, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# use gpu if available

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Always run this — splits are needed even when loading cached features.
splits = load_data()

In [ ]:
# Skip this cell if loading from cached features.
# loading the tokenizer and 4 huggingface models fine-tuned model_135m, model_360m, as well as their reference models: ref_model_135m and ref_model_360m
tokenizer, model_135m, model_360m, ref_model_135m, ref_model_360m = load_models_and_tokenizer(device)

In [ ]:
# training set

train_texts  = splits["train"]["text"].tolist()
train_labels = splits["train"]["is_member"].tolist()

train_135m = process_dataset_in_batches(
    model_135m, tokenizer, train_texts, train_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
train_360m = process_dataset_in_batches(
    model_360m, tokenizer, train_texts, train_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

train_135m.to_csv(f"{SAVE_DIR}train_135m.csv", index=False)
train_360m.to_csv(f"{SAVE_DIR}train_360m.csv", index=False)
print("Train features saved.")

In [ ]:
# validation set
val_texts  = splits["validation"]["text"].tolist()
val_labels = splits["validation"]["is_member"].tolist()

val_135m = process_dataset_in_batches(
    model_135m, tokenizer, val_texts, val_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
val_360m = process_dataset_in_batches(
    model_360m, tokenizer, val_texts, val_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

val_135m.to_csv(f"{SAVE_DIR}val_135m.csv", index=False)
val_360m.to_csv(f"{SAVE_DIR}val_360m.csv", index=False)
print("Validation features saved.")

In [ ]:
# test set
test_texts  = splits["test"]["text"].tolist()
test_labels = [0] * len(test_texts)

test_135m = process_dataset_in_batches(
    model_135m, tokenizer, test_texts, test_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
test_360m = process_dataset_in_batches(
    model_360m, tokenizer, test_texts, test_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

test_135m.to_csv(f"{SAVE_DIR}test_135m.csv", index=False)
test_360m.to_csv(f"{SAVE_DIR}test_360m.csv", index=False)
print("Test features saved.")

In [ ]:
# compile features
features = {
    "train_135m": train_135m,
    "train_360m": train_360m,
    "val_135m":   val_135m,
    "val_360m":   val_360m,
    "test_135m":  test_135m,
    "test_360m":  test_360m,
}

In [ ]:
# Run this cell instead of the extraction cells when resuming a session with saved features.
# import pandas as pd

# features = {
#     "train_135m": pd.read_csv(f"{SAVE_DIR}train_135m.csv"),
#     "train_360m": pd.read_csv(f"{SAVE_DIR}train_360m.csv"),
#     "val_135m":   pd.read_csv(f"{SAVE_DIR}val_135m.csv"),
#     "val_360m":   pd.read_csv(f"{SAVE_DIR}val_360m.csv"),
#     "test_135m":  pd.read_csv(f"{SAVE_DIR}test_135m.csv"),
#     "test_360m":  pd.read_csv(f"{SAVE_DIR}test_360m.csv"),
# }
# print("Features loaded from Drive!")

In [ ]:
# Build classifier using full training set and evaluate on validation set.
results = build_membership_classifier(
    features["train_135m"], features["train_360m"],
    features["val_135m"],   features["val_360m"],
)

In [ ]:
# Inserted at runtime due to issue with the import of zlib

import zlib
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def compute_zlib_baseline(val_df):
    scores = []
    for text in val_df["text"]:
        encoded = text.encode("utf-8", errors="replace")
        ratio = len(zlib.compress(encoded)) / max(len(encoded), 1)
        scores.append(-ratio)  # negate: lower compression = more likely member

    scores = np.array(scores)
    labels = val_df["is_member"].values

    auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    idx = np.argmin(np.abs(fpr - 0.1))
    tpr_at_fpr_01 = tpr[idx]

    print(f"\nZlib Entropy Baseline:")
    print(f"  AUC:         {auc:.4f}")
    print(f"  TPR@FPR=0.1: {tpr_at_fpr_01:.4f}")

    return {"auc": auc, "tpr_at_fpr_01": tpr_at_fpr_01}


zlib_result = compute_zlib_baseline(splits["validation"])

In [ ]:
# Results summary
for name, r in results.items():
    print(f"\n{name}:  AUC={r['auc']:.4f}  TPR@FPR=0.1={r['tpr_at_fpr_01']:.4f}  Acc={r['accuracy']:.4f}")

best_name = max(results, key=lambda x: results[x]["auc"])
best      = results[best_name]
print(f"\nBest: {best_name}  AUC={best['auc']:.4f}  TPR@FPR=0.1={best['tpr_at_fpr_01']:.4f}")

classifier_path = SAVE_DIR + "membership_classifier.pkl"
with open(classifier_path, "wb") as f:
    pickle.dump(best, f)
print(f"Saved classifier to {classifier_path}")

submission_path = "membership_inference_submission.csv"
submission_df = generate_submission(
    features["test_135m"], features["test_360m"],
    best, splits["test"],
    output_path=submission_path,
)
print(submission_df.head())

In [ ]:
# Download Kaggle submission file
from google.colab import files
files.download('membership_inference_submission.csv')